# 检查 DAT 字段是否存在“多填”

目标：检查 `participants.csv` 的 `dat_1`~`dat_10` 是否出现同一字段填了多个中文词的情况（如逗号、分号、顿号、空格等分隔）。

输出包括：
- 总异常数量
- 每个 DAT 字段异常数量
- 异常明细（participant_id、dat_field、raw_value、first_word、word_count）

In [ ]:
import re
from pathlib import Path

import pandas as pd


participants_path = Path("../data/preprocessed/participants.csv")
df = pd.read_csv(participants_path)

# 仅处理 dat_1 ~ dat_10
DAT_COL_PATTERN = re.compile(r"^dat_(10|[1-9])$")
dat_cols = sorted(
    [c for c in df.columns if DAT_COL_PATTERN.match(c)],
    key=lambda x: int(x.split("_")[1]),
)

if not dat_cols:
    raise ValueError("participants.csv 中未找到 dat_1~dat_10 列")


def split_chinese_words(value):
    """提取所有连续中文片段，兼容任意分隔符（逗号、分号、顿号、空格等）。"""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    return re.findall(r"[\u4e00-\u9fff]+", text)


records = []
for _, row in df.iterrows():
    pid = row.get("participant_id")
    for col in dat_cols:
        raw_val = row.get(col)
        words = split_chinese_words(raw_val)
        if len(words) >= 2:
            records.append(
                {
                    "participant_id": pid,
                    "dat_field": col,
                    "raw_value": raw_val,
                    "first_word": words[0],
                    "word_count": len(words),
                }
            )

abnormal_df = pd.DataFrame(records)

print(f"DAT 字段总异常数量: {len(abnormal_df)}")
print(f"存在异常的被试数量: {abnormal_df['participant_id'].nunique() if not abnormal_df.empty else 0}")
print(f"检查字段: {dat_cols}")

if abnormal_df.empty:
    print("未发现 DAT 多填情况。")
else:
    print("\n各 DAT 字段异常数量:")
    display(abnormal_df.groupby("dat_field").size().rename("abnormal_count").reset_index())

    print("\n异常明细（按 participant_id、字段排序）:")
    display(abnormal_df.sort_values(["participant_id", "dat_field"]).reset_index(drop=True))